In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_train.csv
/kaggle/input/competitions/titanic-prediction-scc-study-group/sample submission.csv
/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_test.csv


"PaseengerId"は生存予測に対し無関係なので,indexに引き当てる

In [2]:
train_csv = pd.read_csv("/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_test.csv").set_index("PassengerId")

# 特徴量の追加

前回の「03_eda」より、以下の3つの特徴量"Sex_Pclass","logFare","len_Fam"を追加していく。

In [3]:
train_csv["Sex_Pclass"] = train_csv["Sex"] + "_" + train_csv["Pclass"].astype(str)

train_csv["logFare"] = np.log1p(train_csv["Fare"])

train_csv["Family"] = train_csv[["Parch","SibSp"]].sum(axis=1)
train_csv["len_Fam"] = train_csv["Family"].apply(lambda x:"alone" if x==0 else "basic" if 1<=x<=3 else "large")
train_csv=train_csv.drop("Family",axis=1)

train_csv.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived,Sex_Pclass,logFare,len_Fam
PassengerId,,,,,,,,,,,,,,
693,3,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,NaN,S,1,male_3,4.051712,alone
482,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,NaN,S,0,male_2,0.000000,alone
528,1,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,0,male_1,5.406181,alone
856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,NaN,S,1,female_3,2.336987,basic
802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,NaN,S,1,female_2,3.305054,basic


# testデータにも新特徴量を結合

In [4]:
test_csv["Sex_Pclass"] = test_csv["Sex"] + "_" + test_csv["Pclass"].astype(str)

test_csv["logFare"] = np.log1p(test_csv["Fare"])

test_csv["Family"] = test_csv[["Parch","SibSp"]].sum(axis=1)
test_csv["len_Fam"] = test_csv["Family"].apply(lambda x:"alone" if x==0 else "basic" if 1<=x<=3 else "large")
test_csv=test_csv.drop("Family",axis=1)

test_csv.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Sex_Pclass,logFare,len_Fam
PassengerId,,,,,,,,,,,,,
566,3,"Davies, Mr. Alfred J",male,24.0,2,0,A/4 48871,24.1500,NaN,S,male_3,3.224858,basic
161,3,"Cribb, Mr. John Hatfield",male,44.0,0,1,371362,16.1000,NaN,S,male_3,2.839078,basic
554,3,"Leeni, Mr. Fahim (""Philip Zenni"")",male,22.0,0,0,2620,7.2250,NaN,C,male_3,2.107178,alone
861,3,"Hansen, Mr. Claus Peter",male,41.0,2,0,350026,14.1083,NaN,S,male_3,2.715244,basic
242,3,"Murphy, Miss. Katherine ""Kate""",female,NaN,1,0,367230,15.5000,NaN,Q,female_3,2.803360,basic


# 特徴量を数値列とオブジェクト列に仕分け

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

#目的変数を分離
y = train_csv.Survived
X = train_csv.drop("Survived",axis=1)

#学習用と検証用のデータに分離
X_train_ori,X_valid_ori,y_train,y_valid =\
train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=10)

#低カーディナリティの特徴量の選定(取り扱い安くノイズにならないもの)
cat_cols = [c for c in X_train_ori.columns if X_train_ori[c].nunique()<=10 and X_train_ori[c].dtype == "object"]

#数値列の選定
num_cols = [c for c in X_train_ori.columns if X_train_ori[c].dtype in ["int64","float64"]]

#使用する特徴量の全体
using_cols = cat_cols + num_cols
X_train = X_train_ori[using_cols].copy()
X_valid = X_valid_ori[using_cols].copy()


# 前処理手順の作成

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

#数値列への処理方法
num_transformer = SimpleImputer(strategy = "median")

#カテゴリー列への処理方法
cat_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy = "most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

#それぞれの前処理をまとめる
preprocessor = ColumnTransformer(
    transformers=[
        ("num",num_transformer,num_cols),
        ("cat",cat_transformer,cat_cols)
    ]
)

・数値列に対するstrategy="median"について
>中央値は外れ値の影響を受けないため安定しやすい。現実の数値データは正規分布ではなく歪んでいることが多いので欠損値補完として安定しやすい。
>
・カテゴリー列に対するstrategy="most_frequent"
>欠損値をわからないものとして扱うよりも、最頻値として扱うことでモデルが安定しやすいため

# モデル策定
今回は、ロジスティック回帰(LogisticRegression)を使用

In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)


# 前処理+モデルのパイプラインの作成

In [8]:
from sklearn.metrics import accuracy_score

my_pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",model)
])


#モデルに学習
my_pipeline.fit(X_train,y_train)

#予測の作成
preds = my_pipeline.predict(X_valid)

#予測精度の測定
score = accuracy_score(y_valid,preds)

print(score)

0.7972027972027972


# 提出物の作成

提出用にX,yを分割せずに、全体での予測をだしていく

In [9]:
#X、yの全体のデータを使って学習
my_pipeline.fit(X,y)

#全体データを学習したモデルで予測作成
preds = my_pipeline.predict(test_csv)

output = pd.DataFrame({"PassengerId":test_csv.index,"Survived":preds})
output.to_csv("submission.csv",index=False)